# Silver to Gold KPIs
One row per product category and quarter: total_revenue, order_count, avg_order_value.

In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS adventure_works.gold")

sod = spark.table("adventure_works.silver.sales_order_detail")
prod = spark.table("adventure_works.silver.product").select("ProductID", "ProductLine")

pl = F.upper(F.trim(F.col("ProductLine")))
category = (F.when(pl == "R", "Road")
             .when(pl == "M", "Mountain")
             .when(pl == "S", "Standard")
             .when(pl == "T", "Touring")
             .otherwise("Other"))          # NULL ProductLine -> Other, row is NOT dropped

ts = F.to_timestamp(F.col("ModifiedDate"))  # proxy for order date (no sales_order_header here)
quarter = F.concat(F.year(ts).cast("string"), F.lit("-Q"), F.quarter(ts).cast("string"))

gold = (sod.join(prod, "ProductID", "left")
           .withColumn("category", category)
           .withColumn("quarter", quarter)
           .groupBy("category", "quarter")
           .agg(F.sum(F.col("LineTotal").cast("decimal(19,4)")).alias("total_revenue"),
                F.countDistinct("SalesOrderID").alias("order_count"))
           .withColumn("avg_order_value", F.col("total_revenue") / F.col("order_count")))

(gold.write
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable("adventure_works.gold.sales_kpis_by_category_quarter"))

g = spark.table("adventure_works.gold.sales_kpis_by_category_quarter")
print(f"{g.count()} rows written")
display(g.orderBy(F.desc("total_revenue")).limit(10))

53 rows written


category,quarter,total_revenue,order_count,avg_order_value
Road,2023-Q2,5492956.9679,678,8101.7064423303835
Road,2023-Q3,5190345.6209,694,7478.8841799711816
Road,2024-Q2,4423986.7025,822,5381.9789568126521
Road,2024-Q1,4412414.3437,694,6357.9457402017291
Mountain,2024-Q3,4394856.9081,2381,1845.8029853422932
Mountain,2025-Q1,4318391.5458,2665,1620.4095856660413
Mountain,2024-Q4,4210547.7237,2687,1567.0069682545590
Touring,2025-Q1,4096441.1769,1101,3720.6550198910082
Road,2023-Q4,4042968.5067,670,6034.2813532835821
Road,2024-Q3,4042728.5809,1829,2210.3491420995079


In [0]:
# Checks from the "Run and Verify the Gold KPIs" slide
silver_lines = sod.count()
print("Silver sales lines:", silver_lines, "| Gold rows:", g.count())

bad = g.filter(F.abs(F.col("avg_order_value") - F.col("total_revenue") / F.col("order_count")) > 0.0001).count()
print("Rows where avg_order_value != total_revenue / order_count:", bad)

# Every line still lands in a category: gold revenue must equal silver revenue
silver_rev = sod.agg(F.sum(F.col("LineTotal").cast("decimal(19,4)"))).first()[0]
gold_rev = g.agg(F.sum("total_revenue")).first()[0]
print("Silver revenue:", silver_rev, "| Gold revenue:", gold_rev)
assert bad == 0 and silver_rev == gold_rev, "Gold checks failed"
print("Gold checks passed.")

Silver sales lines: 121317 | Gold rows: 53
Rows where avg_order_value != total_revenue / order_count: 0
Silver revenue: 109846381.4250 | Gold revenue: 109846381.4250
Gold checks passed.
